# Import Modules

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score

# Load the Dataset

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# Data Understanding

In [4]:
train.head()

,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
0,0,0.0,No,6.0,4.0,No,15.0,5.0,Extrovert
1,1,1.0,No,7.0,3.0,No,10.0,8.0,Extrovert
2,2,6.0,Yes,1.0,0.0,NaN,3.0,0.0,Introvert
3,3,3.0,No,7.0,3.0,No,11.0,5.0,Extrovert
4,4,1.0,No,4.0,4.0,No,13.0,NaN,Extrovert


In [5]:
test.duplicated().sum()

0

In [6]:
test.head()

,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency
0,18524,3.0,No,7.0,4.0,No,6.0,NaN
1,18525,NaN,Yes,0.0,0.0,Yes,5.0,1.0
2,18526,3.0,No,5.0,6.0,No,15.0,9.0
3,18527,3.0,No,4.0,4.0,No,5.0,6.0
4,18528,9.0,Yes,1.0,2.0,Yes,1.0,1.0


In [7]:
train.shape

(18524, 9)

In [8]:
test.shape

(6175, 8)

In [9]:
train['Personality'].unique()

array(['Extrovert', 'Introvert'], dtype=object)

In [10]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18524 entries, 0 to 18523
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         18524 non-null  int64  
 1   Time_spent_Alone           17334 non-null  float64
 2   Stage_fear                 16631 non-null  object 
 3   Social_event_attendance    17344 non-null  float64
 4   Going_outside              17058 non-null  float64
 5   Drained_after_socializing  17375 non-null  object 
 6   Friends_circle_size        17470 non-null  float64
 7   Post_frequency             17260 non-null  float64
 8   Personality                18524 non-null  object 
dtypes: float64(5), int64(1), object(3)
memory usage: 1.3+ MB


In [11]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6175 entries, 0 to 6174
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         6175 non-null   int64  
 1   Time_spent_Alone           5750 non-null   float64
 2   Stage_fear                 5577 non-null   object 
 3   Social_event_attendance    5778 non-null   float64
 4   Going_outside              5709 non-null   float64
 5   Drained_after_socializing  5743 non-null   object 
 6   Friends_circle_size        5825 non-null   float64
 7   Post_frequency             5767 non-null   float64
dtypes: float64(5), int64(1), object(2)
memory usage: 386.1+ KB


In [12]:
train.isnull().sum()

id                              0
Time_spent_Alone             1190
Stage_fear                   1893
Social_event_attendance      1180
Going_outside                1466
Drained_after_socializing    1149
Friends_circle_size          1054
Post_frequency               1264
Personality                     0
dtype: int64

In [170]:
test.isnull().sum()

id                             0
Time_spent_Alone             425
Stage_fear                   598
Social_event_attendance      397
Going_outside                466
Drained_after_socializing    432
Friends_circle_size          350
Post_frequency               408
dtype: int64

In [13]:
train['Stage_fear'].unique()

array(['No', 'Yes', nan], dtype=object)

In [14]:
train.columns

Index(['id', 'Time_spent_Alone', 'Stage_fear', 'Social_event_attendance',
       'Going_outside', 'Drained_after_socializing', 'Friends_circle_size',
       'Post_frequency', 'Personality'],
      dtype='object')

# Data Preprocessing

In [15]:
num_cols = ['Time_spent_Alone','Social_event_attendance', 'Going_outside', 'Friends_circle_size' ,'Post_frequency']

cat_cols = ['Stage_fear', 'Drained_after_socializing']

In [16]:
num_transformer = Pipeline(steps=[
    
    ('imputer', SimpleImputer(strategy='median')),
    ('scalar', StandardScaler())

])

In [17]:
cat_transformer = Pipeline(steps=[
    
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))

])

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

In [19]:
# Encode Target (LabelEncoder)

le = LabelEncoder()

y = le.fit_transform(train['Personality']) #  0 = Extrovert, 1 = Introvert


In [20]:
train['Personality'].isnull().sum()

0

In [21]:
y

array([0, 0, 1, ..., 1, 1, 0])

# Model Building

In [22]:
X = train.drop(columns=['Personality', 'id'])

X_test_final = test.drop(columns=['id'])

In [23]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# Models & Parameter Grids

In [ ]:
#searches = []
#
#def run_search(pipe, params, param_dist, name):
#    # Grid Search
#    grid = GridSearchCV(pipe, params, cv=5, scoring="accuracy", n_jobs=-1)
#    grid.fit(X_train, y_train)
#    print(f"\n✅ Finished GridSearch for {name}")
#    print("Best Params:", grid.best_params_)
#    #print("Best CV Score:", round(grid.best_score_, 3))
#
#    # Random Search
#    rand = RandomizedSearchCV(
#        pipe, param_dist, n_iter=20, cv=5,
#        scoring="accuracy", random_state=42, n_jobs=-1
#    )
#    rand.fit(X_train, y_train)
#    print(f"\n✅ Finished RandomSearch for {name}")
#    print("Best Params:", rand.best_params_)
#    #print("Best CV Score:", round(rand.best_score_, 3))
#
#    searches.append((f"{name} (GridSearch)", grid))
#    searches.append((f"{name} (RandomSearch)", rand))

# Hyperparameter Tuning

**Randomized Search**

In [203]:
def run_search_rand(pipe, param_dist, name):

    # Random Search

    rand = RandomizedSearchCV(pipe, param_dist, n_iter=20, cv=5, n_jobs=-1)

    rand.fit(X_train, y_train)

    print(f"Best model for {name}:")
    print("Best Parameters:", rand.best_params_)
    # print("Best Score:", rand.best_score_)
    # print("Best Estimator:", rand.best_estimator_)

    return rand 

**Evaluation Function**

In [193]:
def evaluate_model(search, model_name, X, y, X_valid, y_valid):
    best_model = search.best_estimator_
    val_score = cross_val_score(best_model, X, y, cv=5).mean()
    test_score = accuracy_score(y_valid, best_model.predict(X_valid))

    gap = val_score - test_score
    if gap > 0.05 and val_score >= 0.85:
        fit_msg = "🚨 Overfitting"
    elif val_score < 0.70 and test_score < 0.70:
        fit_msg = "⚠️ Underfitting"
    elif abs(gap) <= 0.05 and test_score >= 0.75:
        fit_msg = "✅ Good Fit"
    else:
        fit_msg = "ℹ️ Borderline"

    print(f"\n--- {model_name} ---")
    print("Best Params:", search.best_params_)
    print("Validation Accuracy:", round(val_score, 3))
    print("Test Accuracy:", round(test_score, 3))
    print("Fit Assessment:", fit_msg)


In [ ]:
# Random Forest
rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])
rf_params = {
    "model__n_estimators": [200, 500, 800],
    "model__max_depth": [None, 10, 20, 30, 40],
    "model__min_samples_split": [2, 4, 6],
    "model__min_samples_leaf": [1, 2, 3]
}
rf_param_dist = {
    "model__n_estimators": np.linspace(200, 1000, 5, dtype=int),
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": np.linspace(2, 10, 5, dtype=int),
    "model__min_samples_leaf": np.linspace(1, 5, 5, dtype=int),
    "model__max_features": ["sqrt", "log2", None]
}

In [ ]:
rf_search = run_search_rand(rf_pipe, rf_param_dist, 'Random Forest')

In [194]:
# Correct function call
evaluate_model(rf_search, "Random Forest", X_train, y_train, X_valid, y_valid)


--- Random Forest ---
Best Params: {'model__n_estimators': 200, 'model__min_samples_split': 4, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': 30}
Validation Accuracy: 0.969
Test Accuracy: 0.968
Fit Assessment: ✅ Good Fit


In [195]:
# XGBoost
xgb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(random_state=42, eval_metric="mlogloss"))
])
xgb_params = {
    "model__n_estimators": [500, 600, 800],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [3, 4, 5, 7],
    "model__subsample": [0.8, 1.0]
}
xgb_param_dist = {
    "model__n_estimators": np.linspace(200, 1000, 5, dtype=int),
    "model__learning_rate": np.linspace(0.01, 0.3, 10),
    "model__max_depth": np.linspace(3, 10, 5, dtype=int),
    "model__subsample": np.linspace(0.7, 1.0, 5),
    "model__colsample_bytree": np.linspace(0.7, 1.0, 5)
}

In [196]:
xgb_search = run_search_rand(xgb_pipe, xgb_param_dist, "XGBoost")

Best model for XGBoost:
Best Parameters: {'model__subsample': 0.7749999999999999, 'model__n_estimators': 600, 'model__max_depth': 4, 'model__learning_rate': 0.01, 'model__colsample_bytree': 0.7}


In [198]:
# Correct function call
evaluate_model(xgb_search, "XGB", X_train, y_train, X_valid, y_valid)


--- XGB ---
Best Params: {'model__subsample': 0.7749999999999999, 'model__n_estimators': 600, 'model__max_depth': 4, 'model__learning_rate': 0.01, 'model__colsample_bytree': 0.7}
Validation Accuracy: 0.969
Test Accuracy: 0.968
Fit Assessment: ✅ Good Fit


In [199]:
# Gradient Boosting
gb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])
gb_params = {
    "model__n_estimators": [200, 300, 500],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [3, 5],
    "model__subsample": [0.8, 1.0]
}
gb_param_dist = {
    "model__n_estimators": np.linspace(100, 500, 5, dtype=int),
    "model__learning_rate": np.linspace(0.01, 0.3, 10),
    "model__max_depth": np.linspace(3, 7, 3, dtype=int),
    "model__subsample": np.linspace(0.7, 1.0, 5)
}

In [201]:
gdb_search = run_search_rand(gb_pipe, gb_param_dist, "Gradient Boosting")

Best model for Gradient Boosting:
Best Parameters: {'model__subsample': 0.925, 'model__n_estimators': 300, 'model__max_depth': 3, 'model__learning_rate': 0.10666666666666666}


In [202]:
# Correct function call
evaluate_model(gdb_search, "GDB", X_train, y_train, X_valid, y_valid)


--- GDB ---
Best Params: {'model__subsample': 0.925, 'model__n_estimators': 300, 'model__max_depth': 3, 'model__learning_rate': 0.10666666666666666}
Validation Accuracy: 0.968
Test Accuracy: 0.969
Fit Assessment: ✅ Good Fit


In [204]:
# LightGBM
lgb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(random_state=42))
])
lgb_params = {
    "model__n_estimators": [200, 400, 500],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [5, 10, 20, 30],
    "model__num_leaves": [31, 52, 63, 127]
}
lgb_param_dist = {
    "model__n_estimators": np.linspace(200, 1000, 5, dtype=int),
    "model__learning_rate": np.linspace(0.01, 0.3, 10),
    "model__max_depth": [-1, 5, 10, 20],
    "model__num_leaves": np.linspace(20, 150, 5, dtype=int),
    "model__subsample": np.linspace(0.7, 1.0, 5)
}

In [205]:
lgb_search = run_search_rand(lgb_pipe, lgb_param_dist, "LightGBM")


[LightGBM] [Info] Number of positive: 3873, number of negative: 10946
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000421 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 67
[LightGBM] [Info] Number of data points in the train set: 14819, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.261354 -> initscore=-1.038945
[LightGBM] [Info] Start training from score -1.038945
Best model for LightGBM:
Best Parameters: {'model__subsample': 0.7, 'model__num_leaves': 52, 'model__n_estimators': 400, 'model__max_depth': 20, 'model__learning_rate': 0.01}


In [206]:
# Correct function call
evaluate_model(lgb_search, "lGB", X_train, y_train, X_valid, y_valid)

[LightGBM] [Info] Number of positive: 3099, number of negative: 8756
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000334 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 67
[LightGBM] [Info] Number of data points in the train set: 11855, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.261409 -> initscore=-1.038660
[LightGBM] [Info] Start training from score -1.038660


d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 3098, number of negative: 8757
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000361 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 67
[LightGBM] [Info] Number of data points in the train set: 11855, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.261324 -> initscore=-1.039097
[LightGBM] [Info] Start training from score -1.039097


d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 3098, number of negative: 8757
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000293 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 67
[LightGBM] [Info] Number of data points in the train set: 11855, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.261324 -> initscore=-1.039097
[LightGBM] [Info] Start training from score -1.039097


d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 3098, number of negative: 8757
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000323 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 67
[LightGBM] [Info] Number of data points in the train set: 11855, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.261324 -> initscore=-1.039097
[LightGBM] [Info] Start training from score -1.039097


d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 3099, number of negative: 8757
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000291 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 67
[LightGBM] [Info] Number of data points in the train set: 11856, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.261387 -> initscore=-1.038774
[LightGBM] [Info] Start training from score -1.038774

--- lGB ---
Best Params: {'model__subsample': 0.7, 'model__num_leaves': 52, 'model__n_estimators': 400, 'model__max_depth': 20, 'model__learning_rate': 0.01}
Validation Accuracy: 0.969
Test Accuracy: 0.968
Fit Assessment: ✅ Good Fit


d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [207]:
# CatBoost
cat_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", CatBoostClassifier(random_state=42, verbose=0))
])
cat_params = {
    "model__iterations": [200, 500, 800],
    "model__depth": [3, 4, 6, 8],
    "model__learning_rate": [0.05, 0.1]
}
cat_param_dist = {
    "model__iterations": np.linspace(200, 1000, 5, dtype=int),
    "model__depth": np.linspace(3, 10, 5, dtype=int),
    "model__learning_rate": np.linspace(0.01, 0.3, 10),
    "model__l2_leaf_reg": np.linspace(1, 10, 5, dtype=int)
}

In [208]:
cat_search = run_search_rand(cat_pipe, cat_param_dist, "CatBoost")


d:\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
3 fits failed out of a total of 100.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "d:\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "d:\anaconda3\Lib\site-packages\sklearn\base.py", line 1363, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 661, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "d:\ana

Best model for CatBoost:
Best Parameters: {'model__learning_rate': 0.07444444444444444, 'model__l2_leaf_reg': 3, 'model__iterations': 800, 'model__depth': 3}


In [209]:
# Correct function call
evaluate_model(cat_search, "lGB", X_train, y_train, X_valid, y_valid)


--- lGB ---
Best Params: {'model__learning_rate': 0.07444444444444444, 'model__l2_leaf_reg': 3, 'model__iterations': 800, 'model__depth': 3}
Validation Accuracy: 0.969
Test Accuracy: 0.969
Fit Assessment: ✅ Good Fit


In [210]:
# Logistic Regression
lr_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=500, random_state=42))
])
lr_params = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__solver": ["lbfgs", "liblinear"]
}
lr_param_dist = {
    "model__C": np.linspace(0.01, 10, 10),
    "model__solver": ["lbfgs", "liblinear"]
}

In [212]:
lr = run_search_rand(lr_pipe, lr_param_dist, "Logistic Regression")


Best model for Logistic Regression:
Best Parameters: {'model__solver': 'liblinear', 'model__C': 5.5600000000000005}


In [213]:
# Correct function call
evaluate_model(lr, "lr", X_train, y_train, X_valid, y_valid)


--- lGB ---
Best Params: {'model__solver': 'liblinear', 'model__C': 5.5600000000000005}
Validation Accuracy: 0.969
Test Accuracy: 0.968
Fit Assessment: ✅ Good Fit


In [214]:
# Decision Tree
dt_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])
dt_params = {
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}
dt_param_dist = {
    "model__max_depth": [None, 5, 10, 20],
    "model__min_samples_split": np.linspace(2, 10, 5, dtype=int),
    "model__min_samples_leaf": np.linspace(1, 5, 5, dtype=int)
}

In [215]:
dt = run_search_rand(dt_pipe, dt_param_dist, "Decision Tree")

Best model for Decision Tree:
Best Parameters: {'model__min_samples_split': 2, 'model__min_samples_leaf': 5, 'model__max_depth': 5}


In [216]:
# Correct function call
evaluate_model(dt, "dt", X_train, y_train, X_valid, y_valid)


--- dt ---
Best Params: {'model__min_samples_split': 2, 'model__min_samples_leaf': 5, 'model__max_depth': 5}
Validation Accuracy: 0.968
Test Accuracy: 0.967
Fit Assessment: ✅ Good Fit


# Chatlink - https://chatgpt.com/c/68aea525-ac68-832a-a5d8-c8c736099dce

**GridSearchCV**

In [26]:
def run_search_grid(pipe, params, name):
    # Grid Search
    grid = GridSearchCV(pipe, params, cv=5, scoring="accuracy", n_jobs=-1)
    grid.fit(X_train, y_train)
    print(f"\n✅ Finished GridSearch for {name}")
    print("Best Params:", grid.best_params_)
    #print("Best CV Score:", round(grid.best_score_, 3))

    return grid

In [29]:
# Random Forest
rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])
rf_params = {
    "model__n_estimators": [200, 500, 800],
    "model__max_depth": [None, 10, 20, 30, 40],
    #"model__min_samples_split": [2, 4, 6],
    #"model__min_samples_leaf": [1, 2, 3]
}

In [30]:
# Run Grid Search for Random Forest
grid_rf = run_search_grid(rf_pipe, rf_params, "Random Forest")



✅ Finished GridSearch for Random Forest
Best Params: {'model__max_depth': 10, 'model__n_estimators': 200}


In [31]:
# XGBoost
xgb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(random_state=42, eval_metric="mlogloss"))
])
xgb_params = {
    "model__n_estimators": [500, 600, 800],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [3, 4, 5, 7],
    #"model__subsample": [0.8, 1.0]
}

In [32]:
grid_xgb = run_search_grid(xgb_pipe, xgb_params, "XGBoost")


✅ Finished GridSearch for XGBoost
Best Params: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 800}


In [33]:
# Gradient Boosting
gb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])
gb_params = {
    "model__n_estimators": [200, 300, 500],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [3, 5],
    #"model__subsample": [0.8, 1.0]
}


In [34]:
grid_gb = run_search_grid(gb_pipe, gb_params, "Gradient Boosting")


✅ Finished GridSearch for Gradient Boosting
Best Params: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}


In [37]:
# LightGBM
lgb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(random_state=42))
])
lgb_params = {
    "model__n_estimators": [200, 400, 500],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [5, 10, 20, 30],
    #"model__num_leaves": [31, 52, 63, 127]
}

In [38]:
grid_lgb = run_search_grid(lgb_pipe, lgb_params, 'LightGBM')

[LightGBM] [Info] Number of positive: 3873, number of negative: 10946
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000896 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 67
[LightGBM] [Info] Number of data points in the train set: 14819, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.261354 -> initscore=-1.038945
[LightGBM] [Info] Start training from score -1.038945

✅ Finished GridSearch for LightGBM
Best Params: {'model__learning_rate': 0.01, 'model__max_depth': 20, 'model__n_estimators': 400}


In [45]:
# CatBoost
cat_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", CatBoostClassifier(random_state=42, verbose=0))
])
cat_params = {
    "model__iterations": [200, 500, 800],
    "model__depth": [3, 4, 6, 8],
    #"model__learning_rate": [0.05, 0.1]
}

In [46]:
grid_cgb = run_search_grid(cat_pipe, cat_params, 'CatBoost')


✅ Finished GridSearch for CatBoost
Best Params: {'model__depth': 3, 'model__iterations': 200}


In [41]:
# Decision Tree
dt_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])
dt_params = {
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5, 10],
    #"model__min_samples_leaf": [1, 2, 4]
}

In [42]:
grid_dt = run_search_grid(dt_pipe, dt_params, "Decision Tree")


✅ Finished GridSearch for Decision Tree
Best Params: {'model__max_depth': 5, 'model__min_samples_split': 10}


In [43]:
# Logistic Regression
lr_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=500, random_state=42))
])
lr_params = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__solver": ["lbfgs", "liblinear"]
}

In [44]:
grid_lr = run_search_grid(lr_pipe, lr_params, "Logistic Regression")


✅ Finished GridSearch for Logistic Regression
Best Params: {'model__C': 10, 'model__solver': 'lbfgs'}


**Evaluation Fuction**

In [51]:
# Evaluation Fuction

def evaluate_model(search, model_name, X, y, X_valid, y_valid):
    best_model = search.best_estimator_
    val_score = cross_val_score(best_model, X, y, cv=5).mean()
    test_score = accuracy_score(y_valid, best_model.predict(X_valid))

    gap = val_score - test_score
    if gap > 0.05 and val_score >= 0.85:
        fit_msg = "🚨 Overfitting"
    elif val_score < 0.70 and test_score < 0.70:
        fit_msg = "⚠️ Underfitting"
    elif abs(gap) <= 0.05 and test_score >= 0.75:
        fit_msg = "✅ Good Fit"
    else:
        fit_msg = "ℹ️ Borderline"

    print(f"\n--- {model_name} ---")
    print("Best Params:", search.best_params_)
    print("Validation Accuracy:", round(val_score, 3))
    print("Test Accuracy:", round(test_score, 3))
    print("Fit Assessment:", fit_msg)

    return {
        "Model": model_name,
        "Validation_Accuracy": round(val_score, 3),
        "Test_Accuracy": round(test_score, 3),
        "Fit_Assessment": fit_msg
    }

# Store model name and fitted GridSearchCV object

In [52]:
grids = [
    ("Random Forest", grid_rf),
    ("Decision Tree", grid_dt),
    ("Logistic Regression", grid_lr),
    ("Gradient Boosting", grid_gb),
    ("XGBoost", grid_xgb),
    ("LightGBM", grid_lgb),
    ("CatBoost", grid_cgb)
]

results = []
# Loop through each model and collect results
for name, grid in grids:
    res = evaluate_model(grid, name, X_train, y_train, X_valid, y_valid)
    results.append(res)

# Convert results to DataFrame
df_results = pd.DataFrame(results)


--- Random Forest ---
Best Params: {'model__max_depth': 10, 'model__n_estimators': 200}
Validation Accuracy: 0.969
Test Accuracy: 0.968
Fit Assessment: ✅ Good Fit

--- Decision Tree ---
Best Params: {'model__max_depth': 5, 'model__min_samples_split': 10}
Validation Accuracy: 0.968
Test Accuracy: 0.968
Fit Assessment: ✅ Good Fit

--- Logistic Regression ---
Best Params: {'model__C': 10, 'model__solver': 'lbfgs'}
Validation Accuracy: 0.969
Test Accuracy: 0.968
Fit Assessment: ✅ Good Fit

--- Gradient Boosting ---
Best Params: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}
Validation Accuracy: 0.969
Test Accuracy: 0.968
Fit Assessment: ✅ Good Fit

--- XGBoost ---
Best Params: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 800}
Validation Accuracy: 0.969
Test Accuracy: 0.969
Fit Assessment: ✅ Good Fit
[LightGBM] [Info] Number of positive: 3099, number of negative: 8756
[LightGBM] [Info] Auto-choosing row-wise multi-threading,

d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 3098, number of negative: 8757
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000633 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 67
[LightGBM] [Info] Number of data points in the train set: 11855, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.261324 -> initscore=-1.039097
[LightGBM] [Info] Start training from score -1.039097


d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 3098, number of negative: 8757
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000389 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 67
[LightGBM] [Info] Number of data points in the train set: 11855, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.261324 -> initscore=-1.039097
[LightGBM] [Info] Start training from score -1.039097


...

[LightGBM] [Info] Number of positive: 3098, number of negative: 8757
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000377 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 67
[LightGBM] [Info] Number of data points in the train set: 11855, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.261324 -> initscore=-1.039097
[LightGBM] [Info] Start training from score -1.039097


d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 3099, number of negative: 8757
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000426 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 67
[LightGBM] [Info] Number of data points in the train set: 11856, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.261387 -> initscore=-1.038774
[LightGBM] [Info] Start training from score -1.038774


d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- LightGBM ---
Best Params: {'model__learning_rate': 0.01, 'model__max_depth': 20, 'model__n_estimators': 400}
Validation Accuracy: 0.969
Test Accuracy: 0.969
Fit Assessment: ✅ Good Fit

--- CatBoost ---
Best Params: {'model__depth': 3, 'model__iterations': 200}
Validation Accuracy: 0.969
Test Accuracy: 0.968
Fit Assessment: ✅ Good Fit


In [53]:
df_results

,Model,Validation_Accuracy,Test_Accuracy,Fit_Assessment
0,Random Forest,0.969,0.968,✅ Good Fit
1,Decision Tree,0.968,0.968,✅ Good Fit
2,Logistic Regression,0.969,0.968,✅ Good Fit
3,Gradient Boosting,0.969,0.968,✅ Good Fit
4,XGBoost,0.969,0.969,✅ Good Fit
5,LightGBM,0.969,0.969,✅ Good Fit
6,CatBoost,0.969,0.968,✅ Good Fit


# Final Prediction & Submission

In [64]:
best_lgb_model = grid_lgb.best_estimator_

test_predictions = best_lgb_model.predict(X_test_final)

d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [100]:
test_predictions.info()

<class 'pandas.core.series.Series'>
RangeIndex: 6175 entries, 0 to 6174
Series name: None
Non-Null Count  Dtype 
--------------  ----- 
6175 non-null   object
dtypes: object(1)
memory usage: 48.4+ KB


In [93]:
# Example: create a submission DataFrame

submission = pd.DataFrame({
    "id": test["id"],
    "Personality": test_predictions
})

In [94]:
submission['Personality'] = submission['Personality'].astype(int).map({0: 'Extrovert', 1: 'Introvert'})
# submission['Personality'] = submission['Personality'].map({'0': 'Extrovert', '1': 'Introvert'})



In [95]:
submission.head()

,id,Personality
0,18524,Extrovert
1,18525,Introvert
2,18526,Extrovert
3,18527,Extrovert
4,18528,Introvert


In [98]:
submission.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6175 entries, 0 to 6174
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           6175 non-null   int64 
 1   Personality  6175 non-null   object
dtypes: int64(1), object(1)
memory usage: 96.6+ KB


In [99]:
# Save submission
submission.to_csv('submission.csv', index=False)
print("✅ Predictions saved to submission.csv")

✅ Predictions saved to submission.csv


In [69]:
sample = pd.read_csv('sample_submission.csv')

In [72]:
sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6175 entries, 0 to 6174
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           6175 non-null   int64 
 1   Personality  6175 non-null   object
dtypes: int64(1), object(1)
memory usage: 96.6+ KB


In [75]:
sample.head()

,id,Personality
0,18524,Extrovert
1,18525,Extrovert
2,18526,Extrovert
3,18527,Extrovert
4,18528,Extrovert
